# Haystack

**Domain:** Agentic AI  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **Haystack 2.x** (by [deepset](https://www.deepset.ai/)) — an open-source Python framework for building **LLM applications as pipelines of composable components**. Its signature move: you assemble a search/RAG/agent app by wiring small, typed **Components** (retrievers, prompt builders, generators, custom functions) into a **Pipeline** — a directed graph with explicit named input/output sockets — instead of writing the orchestration glue by hand.

If you know [[langchain]]'s LCEL or [[llamaindex-agents]]' query engines, Haystack occupies the same space but leads with an **explicit, inspectable, serializable pipeline graph**. It started as a production search/RAG framework, so its retrieval and document-store ergonomics are first-class. Compare with [[langchain]], [[llamaindex-agents]], and [[langgraph]] for the same problem in other framings.

## 1. What & Why

An LLM app is rarely one model call. A real RAG app *embeds a query, retrieves chunks, ranks them, stuffs them into a prompt template, calls a model, and post-processes the answer* — and a search app, an extractive-QA app, or an agent each chain a different sequence of those steps. Write that glue by hand and you get a tangle of function calls where the data flow, the types, and the failure points are all implicit.

**Haystack makes the data flow the first-class object.** You build small **Components** (each a class with a typed `run()` method) and connect them into a **Pipeline** — a directed graph. Haystack validates at connection time that every wire links a compatible output socket to an input socket, runs the components in dependency order, and passes data between them. The graph is **inspectable** (draw it, list its sockets) and **serializable** (dump the whole app to YAML, reload it elsewhere).

**The problem it solves:**

- **Production RAG / search** — Haystack grew up as a search framework, so document stores, BM25 + embedding retrievers, rankers, and readers are mature and batteries-included.
- **Composable, swappable steps** — swap an `InMemoryDocumentStore` for Elasticsearch, or OpenAI for a local model, by changing one component; the rest of the graph is untouched.
- **Explicit, debuggable flow** — connections are typed and checked up front, so a mismatched pipeline fails at build time with a clear error, not deep in a run.
- **Deployability** — pipelines serialize to YAML and serve over REST via [Hayhook](https://github.com/deepset-ai/hayhook), so the same graph you prototype is the artifact you ship.
- **Agents too** — Haystack 2.x ships an `Agent` component and tool-calling, so a tool-using loop is just another node in a pipeline.

**When to reach for it:** you're building a search/RAG/QA app (or an agent over your data) and want an explicit, swappable, deployable pipeline rather than imperative glue. **When *not* to:** a single one-shot LLM call needs no pipeline — call the SDK directly; the graph machinery only pays off once you have several steps to wire and swap.

## 2. Mental Model

**A Haystack Pipeline is a circuit board; Components are chips with labeled pins.**

Each Component is a chip: it declares typed **input sockets** (the pins it consumes) and **output sockets** (the pins it produces). You solder them together by connecting a named output pin to a named input pin. Haystack checks the pins are type-compatible *before* you power it on, then runs the chips in dependency order, carrying each chip's output along the wires to the next.

```
   query ─────────────┐
                       ▼
            ┌────────────────────┐
            │   BM25Retriever    │   in:  query
            │  (over a doc store)│   out: documents
            └─────────┬──────────┘
        documents     │  (output socket -> input socket)
                      ▼
            ┌────────────────────┐
   question ──────▶ │   PromptBuilder    │   in: documents, question
            │  (Jinja template)  │   out: prompt
            └─────────┬──────────┘
        prompt        │
                      ▼
            ┌────────────────────┐
            │  OpenAIGenerator   │   in:  prompt
            │   (the LLM call)   │   out: replies
            └─────────┬──────────┘
                      ▼
                   answer
```

Two ideas do all the work:

- **Component** — a unit of computation. A class decorated with `@component` whose `run()` has type-hinted args (input sockets) and an `@component.output_types(...)`-declared dict return (output sockets). Stateless config in `__init__`, data flow through `run()`.
- **Pipeline** — the board. `add_component(name, instance)` places a chip; `connect("a.out", "b.in")` wires a pin; `run({...})` feeds the entry components and returns the leaf outputs. The graph is a DAG (loops are allowed for agent/retry patterns, but data only flows along declared wires).

Everything else — document stores, retrievers, generators, the `Agent` — is just a library of pre-built chips you drop onto the board.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **Component** | A class decorated with `@component` exposing a typed `run()`. Its parameter type hints define **input sockets**; `@component.output_types(name=Type, ...)` declares **output sockets**. The unit you connect. |
| **Pipeline** | The orchestrator/DAG. `add_component`, `connect("comp.out", "comp.in")`, then `run(inputs)`. Validates socket types at connect time and runs components in dependency order. |
| **Socket** | A named, typed input or output port on a component. Connections are `producer.output_socket → consumer.input_socket`; types must be compatible or `connect()` raises. |
| **Document** | Haystack's data container: `content`, plus `meta`, `embedding`, `score`, `id`. The currency that flows through retrievers, rankers, and readers. |
| **DocumentStore** | Where Documents live. `InMemoryDocumentStore` for dev/tests; `Elasticsearch`, `OpenSearch`, `Weaviate`, `Pinecone`, `Qdrant`, `pgvector` for production (separate integration packages). |
| **Retriever** | Pulls candidate Documents for a query. `InMemoryBM25Retriever` (keyword, no embeddings) vs `...EmbeddingRetriever` (vector similarity). Bound to a specific store. |
| **PromptBuilder** | Renders a **Jinja2** template into a prompt string, looping over retrieved `documents` and injecting variables — the bridge from retrieval to generation. |
| **Generator** | The LLM call. `OpenAIGenerator` / `OpenAIChatGenerator`, plus Anthropic, Cohere, Hugging Face, Ollama, etc. (integration packages). Takes a prompt, returns `replies`. |
| **Agent** | A built-in component that runs a **tool-calling loop** around a chat generator — Haystack's answer to [[react]]-style agents, usable standalone or as a node in a larger pipeline. |
| **Serialization** | `pipeline.dumps()` / `Pipeline.loads()` (YAML). The whole app is data, so you can version, deploy, and serve it (via Hayhook) unchanged. |

## 4. Setup

Haystack 2.x is the `haystack-ai` package (note: the old 1.x `farm-haystack` is a different, deprecated line — don't mix them):

```bash
pip install haystack-ai                 # core: components, Pipeline, InMemory store/retrievers
# Integrations are separate packages, installed only as needed:
pip install elasticsearch-haystack      # production document store
pip install sentence-transformers       # local embedding models (no API key)
```

The core package alone ships the `InMemoryDocumentStore`, BM25 retrievers, `PromptBuilder`, and the `Pipeline` machinery — so **Examples 1 and 2 below run offline, no API key, no model download.** Generators that hit a hosted model (`OpenAIGenerator`, etc.) read their key from the environment:

```python
import os; os.environ["OPENAI_API_KEY"] = "sk-..."
from haystack.components.generators import OpenAIGenerator
generator = OpenAIGenerator(model="gpt-4o-mini")   # key picked up from env
```

**Example 3** drives a real LLM and is gated behind an `os.getenv` check, so the notebook executes top-to-bottom either way. The cell below reports what's available.

In [ ]:
# Environment check — Examples 1 & 2 need only haystack-ai (offline, no key).
import importlib.util as u, os

def installed(name: str) -> bool:
    try:
        return u.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

has_haystack = installed("haystack")
has_key      = bool(os.getenv("OPENAI_API_KEY"))

if has_haystack:
    import haystack
    print("haystack-ai:", haystack.__version__)
else:
    print("haystack-ai: NOT installed  ->  pip install haystack-ai")

print("OPENAI_API_KEY present:", has_key)
print()
print("Examples 1 & 2 run on the core package alone (offline, deterministic).")
print("Example 3 calls a real LLM only if OPENAI_API_KEY is set; otherwise it prints the call shape.")

## 5. Worked Examples

### Example 1 — Documents, a store, and a retriever

The atoms of a Haystack search app are the **Document**, the **DocumentStore** that holds them, and the **Retriever** that pulls relevant ones for a query. Here we load a handful of Documents into an `InMemoryDocumentStore` and run a `InMemoryBM25Retriever` — pure keyword (BM25) scoring, so it needs **no embeddings, no model, no network**. Note the retriever returns scored Documents ranked by relevance: this is the "retrieve" half of RAG, standing alone.

In [ ]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

store = InMemoryDocumentStore()
store.write_documents([
    Document(content="Haystack is an open-source framework by deepset for building LLM and RAG applications."),
    Document(content="A Pipeline connects Components via named, typed input and output sockets into a DAG."),
    Document(content="The InMemoryBM25Retriever ranks documents by BM25 keyword matching, no embeddings needed."),
    Document(content="Paris is the capital of France and sits on the river Seine."),
])
print("documents in store:", store.count_documents())

# A Retriever is itself a Component; calling .run() returns a dict keyed by its output socket.
retriever = InMemoryBM25Retriever(document_store=store)
result = retriever.run(query="What is Haystack?", top_k=2)

print("\ntop matches for 'What is Haystack?':")
for doc in result["documents"]:
    print(f"  score={doc.score:.3f}  {doc.content[:60]}...")

### Example 2 — Wiring a Pipeline (retriever → prompt builder)

Now the defining Haystack move: connect components into a **Pipeline**. We add the retriever and a `PromptBuilder` (a Jinja2 template), then `connect()` the retriever's `documents` output socket to the prompt builder's `documents` input socket. When we `run()` the pipeline, Haystack feeds each entry component its inputs, flows the retrieved documents along the wire, and the prompt builder renders a grounded prompt — the exact string you'd hand to an LLM. This is the full **retrieve-then-prompt** front half of RAG, with the LLM call (Example 3) deliberately left as the last swappable node.

In [ ]:
from haystack import Pipeline
from haystack.components.builders import PromptBuilder

template = """Answer the question using ONLY the context below.
Context:
{% for doc in documents %}- {{ doc.content }}
{% endfor %}
Question: {{ question }}
Answer:"""

pipe = Pipeline()
pipe.add_component("retriever", InMemoryBM25Retriever(document_store=store))
pipe.add_component("prompt", PromptBuilder(template=template, required_variables=["question"]))

# Wire output socket -> input socket. Haystack type-checks this connection at build time.
pipe.connect("retriever.documents", "prompt.documents")

question = "What is Haystack?"
out = pipe.run({
    "retriever": {"query": question, "top_k": 2},
    "prompt":    {"question": question},
})
print(out["prompt"]["prompt"])

### Example 3 — Add a Generator to complete the RAG pipeline (gated)

The only thing missing from Example 2 is the LLM call. Adding it is one more node: drop an `OpenAIGenerator` onto the board and connect `prompt.prompt → llm.prompt`. The retriever and prompt builder are **unchanged** — that's the payoff of the pipeline model. This cell runs the real LLM only if `OPENAI_API_KEY` is set; otherwise it prints the canonical wiring so the notebook still executes top-to-bottom.

In [ ]:
if has_haystack and has_key:
    from haystack.components.generators import OpenAIGenerator

    rag = Pipeline()
    rag.add_component("retriever", InMemoryBM25Retriever(document_store=store))
    rag.add_component("prompt", PromptBuilder(template=template, required_variables=["question"]))
    rag.add_component("llm", OpenAIGenerator(model="gpt-4o-mini"))   # reads OPENAI_API_KEY from env
    rag.connect("retriever.documents", "prompt.documents")
    rag.connect("prompt.prompt", "llm.prompt")                       # the one new wire

    q = "What is Haystack and what is a Pipeline?"
    answer = rag.run({"retriever": {"query": q, "top_k": 3}, "prompt": {"question": q}})
    print(answer["llm"]["replies"][0])
else:
    print("Skipping live LLM run (set OPENAI_API_KEY to enable).")
    print("With a key set, you add exactly two lines to the Example 2 pipeline:")
    print()
    print('    rag.add_component("llm", OpenAIGenerator(model="gpt-4o-mini"))')
    print('    rag.connect("prompt.prompt", "llm.prompt")')
    print()
    print("    answer = rag.run({'retriever': {'query': q}, 'prompt': {'question': q}})")
    print("    answer['llm']['replies'][0]   # -> the grounded answer string")

### Example 4 — A custom Component

When no built-in chip fits, you write one. A Component is just a class decorated with `@component` whose `run()` has type-hinted parameters (its input sockets) and an `@component.output_types(...)` decorator declaring its output sockets. Once decorated, your class drops onto any Pipeline board and connects to the rest exactly like a built-in. Here is a trivial one to show the whole contract in a few lines — no LLM, fully offline.

In [ ]:
from haystack import component

@component
class KeywordTagger:
    """Tag a document's content with any of a watchlist of keywords it contains."""

    def __init__(self, keywords: list[str]):
        self.keywords = [k.lower() for k in keywords]

    @component.output_types(tags=list[str], hit_count=int)   # declares two output sockets
    def run(self, text: str):                                # 'text' is the input socket
        found = [k for k in self.keywords if k in text.lower()]
        return {"tags": found, "hit_count": len(found)}

tagger = KeywordTagger(keywords=["Haystack", "Pipeline", "France", "embeddings"])
print(tagger.run(text="Haystack builds a Pipeline of components for RAG."))
print(tagger.run(text="Paris is the capital of France."))

## 6. Gotchas & Pitfalls

- **`haystack-ai` (2.x) vs `farm-haystack` (1.x).** They are different packages with incompatible APIs. Every current tutorial and the components in this notebook are 2.x (`pip install haystack-ai`). Installing both in one environment, or following a 1.x guide against 2.x, breaks imports in confusing ways. Pin `haystack-ai`.
- **Connecting by component, not by socket.** `connect("retriever", "prompt")` only works if there's a single unambiguous output→input match. The moment a component has multiple sockets you must be explicit: `connect("retriever.documents", "prompt.documents")`. Be explicit by default — it documents the graph and avoids surprise mismatches.
- **Socket type mismatches fail at connect time (good).** Haystack validates that an output socket's type fits the input socket when you call `connect()`, raising immediately. That's a feature — but it means you must declare `@component.output_types(...)` and real parameter hints on custom components, or the type checker has nothing to match.
- **`run()` inputs are nested by component name.** `pipeline.run({"retriever": {"query": q}, "prompt": {"question": q}})` — the top-level keys are *component names*, not socket names. Passing a flat dict, or feeding a value to a socket that's already wired from another component, is a common first-run error.
- **Forgetting the retriever is bound to a store.** A retriever holds a reference to one `DocumentStore`. Write your documents *before* you run, and make sure the retriever points at the store you actually populated — an empty store silently returns no documents, not an error.
- **BM25 ≠ semantic search.** `InMemoryBM25Retriever` is keyword matching: great offline and for exact terms, but it won't match paraphrases. For semantic recall you need an embedding retriever plus a document embedder (and a model/API), which adds setup and cost.
- **`InMemoryDocumentStore` is dev-only.** It lives in RAM and vanishes with the process. For anything persistent or large, swap in a real store (Elasticsearch, Qdrant, pgvector, …) — but those are separate integration packages with their own services.
- **Agent/looping pipelines can run away.** The `Agent` component and any cyclic pipeline run a tool-calling loop; like any [[react]] agent, set max-iteration limits and watch token spend, or a confused model loops indefinitely.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs Haystack |
|---|---|---|
| **Haystack (this)** | Production **search / RAG / QA** apps you want as an explicit, swappable, **serializable pipeline**; strong document-store & retriever ecosystem | Most batteries-included retrieval stack; pipeline graph is inspectable and deployable. Heavier abstraction than calling an SDK directly for a one-shot |
| **Direct SDK call** (OpenAI/Anthropic) | A single prompt → completion, no retrieval, no multi-step flow | Zero framework overhead, but you hand-roll any retrieval, templating, or chaining. Use until you have ≥2 steps to wire |
| **[[langchain]] / [[langgraph]]** | Broadest integration catalog; LangGraph for explicit **stateful agent state-machines** with persistence | Larger ecosystem and graph-state control; Haystack's retrieval ergonomics and build-time socket type-checking are cleaner for pure RAG |
| **[[llamaindex-agents]]** | RAG where the **agent routes across multiple indexes**; data-framework ergonomics (`QueryEngineTool`) | LlamaIndex leads with indexes/query engines; Haystack leads with an explicit component graph. Both excel at RAG, different mental models |
| **Hand-rolled glue** | Tiny scripts, full control, no dependency | No type-checked wiring, no serialization, no swappable components — fine until the flow grows, then it rots into spaghetti |

**Rule of thumb:** choose Haystack when your app is a **multi-step retrieval/RAG/search flow** and you value an explicit, type-checked, deployable pipeline of swappable parts. Drop to a direct SDK call for one-shots; reach for LangGraph when heavy **stateful agent orchestration** (not retrieval) is the centre of gravity; reach for LlamaIndex when **routing across many indexes** is the core problem.

## 8. Resources

- **Haystack docs (official, 2.x)** — https://docs.haystack.deepset.ai/docs/intro
- **Haystack tutorials (hands-on, runnable Colabs)** — https://haystack.deepset.ai/tutorials
- **Creating a custom Component (the `@component` contract)** — https://docs.haystack.deepset.ai/docs/custom-components
- **Pipelines — concepts, connecting sockets, serialization** — https://docs.haystack.deepset.ai/docs/pipelines
- **Agents & tool calling in Haystack 2.x** — https://docs.haystack.deepset.ai/docs/agent
- **Haystack on GitHub (source, integrations, changelog)** — https://github.com/deepset-ai/haystack

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def output_types(**sockets):
    ...


class Pipeline:
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE